# Edmonton Geographic Heatmap Tool

Visualize property data as weighted geographic heatmaps over Edmonton.

**Two visualization modes:**
1. **Coordinate Heatmap** — density heatmap based on property lat/lon, weighted by a selected column
2. **Submarket Choropleth** — Edmonton submarket boundary polygons filled by aggregated weights

**Flexible weights:** Change `WEIGHT_COLUMN` to any numeric column — unit counts, vacancy rates, rent PSF, absorption ratios, cap rates, custom ratios, etc.

---
**Quick start:**
1. Run the install cell (first time only)
2. Set `WEIGHT_COLUMN` in **Step 1 – Configuration**
3. Replace sample data with your actual data in **Step 2 – Data Input**
4. `Runtime → Run all`

In [ ]:
%%capture
!pip install folium geopandas pandas numpy requests shapely plotly
print('All packages ready.')

In [ ]:
import folium
from folium.plugins import HeatMap
import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import json
from shapely.geometry import Point, Polygon
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

---
## Step 1 – Configuration

Edit the variables below to control which column drives the heatmap intensity and how the maps look.

In [ ]:
# ============================================================
# CONFIGURATION  —  edit these values
# ============================================================

# Which column to use as heatmap weight.
# Any numeric column works: 'units', 'vacancy_rate', 'rent_psf',
# 'absorption_rate', 'cap_rate', or any custom ratio column.
WEIGHT_COLUMN = 'units'

# Normalization before passing weights to the heatmap renderer.
#   'minmax' — scales values to [0, 1]  (default, works for most cases)
#   'log'    — log scale; useful when values are heavily right-skewed
NORMALIZE_METHOD = 'minmax'

# Aggregation used when rolling up properties into submarket polygons.
#   'sum'   — total weight in the submarket  (good for unit counts)
#   'mean'  — average weight                 (good for rates/ratios)
#   'count' — number of properties           (ignores weight column)
CHOROPLETH_AGGREGATION = 'sum'

# Visualization mode:
#   'heatmap'    — coordinate-based density heatmap only
#   'choropleth' — submarket boundary fill only
#   'both'       — show both maps (default)
VISUALIZATION_MODE = 'both'

# Edmonton city centre [lat, lon] and default zoom
MAP_CENTER  = [53.5461, -113.4938]
ZOOM_START  = 11

# Heatmap appearance
HEATMAP_RADIUS      = 35    # pixel radius of each point's influence
HEATMAP_BLUR        = 20    # blur amount (higher = smoother)
HEATMAP_MIN_OPACITY = 0.3
HEATMAP_GRADIENT    = {0.2: 'blue', 0.45: 'lime', 0.70: 'yellow', 1.0: 'red'}

# Choropleth colour scale
# Options: 'YlOrRd', 'Blues', 'RdYlGn', 'Purples', 'BuGn', 'OrRd'
CHOROPLETH_COLOR = 'YlOrRd'

# Map tile style
# Options: 'CartoDB positron', 'CartoDB dark_matter', 'OpenStreetMap', 'Stamen Toner'
MAP_TILE = 'CartoDB positron'

print('Configuration set.')
print(f'  Weight column       : {WEIGHT_COLUMN}')
print(f'  Normalization       : {NORMALIZE_METHOD}')
print(f'  Choropleth agg      : {CHOROPLETH_AGGREGATION}')
print(f'  Visualization mode  : {VISUALIZATION_MODE}')

---
## Step 2 – Data Input

Replace the sample data below with your actual property data.

**Required columns:**
| Column | Description |
|--------|-------------|
| `latitude` | Decimal degrees, WGS84 |
| `longitude` | Decimal degrees, WGS84 |
| your weight column | Any numeric value (units, rates, ratios…) |

**Option A** — edit `sample_data` directly in the next cell  
**Option B** — load from a CSV file (instructions in the cell below)

In [ ]:
# ============================================================
# OPTION A: define data directly (edit values or replace entirely)
# ============================================================
sample_data = {
    'property_id': [
        'P001','P002','P003','P004','P005',
        'P006','P007','P008','P009','P010',
        'P011','P012','P013','P014','P015',
        'P016','P017','P018','P019','P020'
    ],
    'property_name': [
        'Churchill Place',      'Oliver Square Apts',  'Whyte Ave Flats',
        'West End Tower',       'Mill Woods Heights',  'Windermere Walk',
        'Castle Downs Commons', 'Clareview Court',     'Glenora Manor',
        'South Common Suites',  'Downtown Lofts',      'Strathcona Flats',
        'Griesbach Village',    'McConachie Homes',    'Terwillegar Park',
        'Meadows Landing',      'Hollick-Kenyon',      'Callingwood Towers',
        'Tamarack Ridge',       'Heritage Park Apts'
    ],
    'latitude': [
        53.5461, 53.5445, 53.5225, 53.5300, 53.4750,
        53.4350, 53.6225, 53.5975, 53.5550, 53.4625,
        53.5500, 53.5200, 53.6100, 53.6350, 53.4600,
        53.4800, 53.6400, 53.5250, 53.4700, 53.4450
    ],
    'longitude': [
        -113.4938, -113.5200, -113.5012, -113.5800, -113.4612,
        -113.5912, -113.5100, -113.4212, -113.5350, -113.5012,
        -113.4850, -113.4900, -113.5200, -113.4500, -113.5800,
        -113.4200, -113.4700, -113.6235, -113.4100, -113.5700
    ],
    # ── Weight columns ──────────────────────────────────────────
    # Number of residential units per property
    'units': [
        250, 180, 120, 320, 210,
         95, 175, 140,  88, 265,
        310, 155, 130, 200,  85,
        190, 145, 280, 165, 110
    ],
    # Physical vacancy rate  (ratio: 0.0 – 1.0)
    'vacancy_rate': [
        0.08, 0.12, 0.05, 0.15, 0.09,
        0.03, 0.11, 0.07, 0.04, 0.13,
        0.10, 0.06, 0.08, 0.14, 0.02,
        0.09, 0.06, 0.16, 0.07, 0.05
    ],
    # Rent per square foot  ($/psf)
    'rent_psf': [
        2.80, 2.50, 2.90, 2.60, 2.10,
        3.20, 2.00, 1.95, 3.10, 2.40,
        3.00, 2.75, 2.15, 2.05, 3.30,
        2.20, 2.10, 2.55, 2.25, 3.05
    ],
    # Absorption rate  (units leased / units available, ratio: 0.0 – 1.0)
    'absorption_rate': [
        0.92, 0.88, 0.95, 0.85, 0.91,
        0.97, 0.89, 0.93, 0.96, 0.87,
        0.90, 0.94, 0.92, 0.86, 0.98,
        0.91, 0.94, 0.84, 0.93, 0.95
    ]
}

df = pd.DataFrame(sample_data)

# ============================================================
# OPTION B: load from CSV instead  (comment out Option A above)
# ============================================================
# from google.colab import files
# uploaded = files.upload()                  # pick your file in the dialog
# df = pd.read_csv(list(uploaded.keys())[0])
#
# — OR — if you already have the file in Colab:
# df = pd.read_csv('/content/your_properties.csv')
#
# Required columns: 'latitude', 'longitude', and whatever WEIGHT_COLUMN is set to.

print(f'Loaded {len(df)} properties.')
df.head()

In [ ]:
# ── Data validation ─────────────────────────────────────────
required = ['latitude', 'longitude', WEIGHT_COLUMN]
missing  = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}. '
                     f'Available columns: {list(df.columns)}')

df = df.dropna(subset=required).copy()
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df[WEIGHT_COLUMN] = pd.to_numeric(df[WEIGHT_COLUMN], errors='coerce')
df = df.dropna(subset=required)

print(f'Valid properties  : {len(df)}')
print(f'Weight column     : "{WEIGHT_COLUMN}"')
print()
print(df[WEIGHT_COLUMN].describe().rename('stats').to_frame())
print()
print(f'Latitude  range   : {df["latitude"].min():.4f}  →  {df["latitude"].max():.4f}')
print(f'Longitude range   : {df["longitude"].min():.4f}  →  {df["longitude"].max():.4f}')

---
## Step 3 – Coordinate-Based Heatmap

Each property is placed at its lat/lon. The colour intensity at each point reflects the **normalized weight** of that property. Warmer colours (yellow → red) = higher weight.

In [ ]:
def normalize_weights(series, method='minmax'):
    """Normalize a numeric Series to [0, 1] for heatmap rendering."""
    s = series.astype(float).copy()
    if method == 'log':
        s = np.log1p(s)
    lo, hi = s.min(), s.max()
    if hi > lo:
        return (s - lo) / (hi - lo)
    return pd.Series(np.ones(len(s)), index=s.index)


def create_coordinate_heatmap(df, weight_col,
                               center=MAP_CENTER, zoom=ZOOM_START):
    """
    Build a folium map with a weighted density heatmap.

    Parameters
    ----------
    df         : DataFrame — must contain 'latitude', 'longitude', weight_col
    weight_col : str       — column used as heatmap intensity
    center     : [lat, lon]
    zoom       : int
    """
    m = folium.Map(location=center, zoom_start=zoom, tiles=MAP_TILE)

    # Heatmap layer
    weights   = normalize_weights(df[weight_col], NORMALIZE_METHOD)
    heat_data = list(zip(df['latitude'], df['longitude'], weights))
    HeatMap(
        heat_data,
        radius=HEATMAP_RADIUS,
        blur=HEATMAP_BLUR,
        min_opacity=HEATMAP_MIN_OPACITY,
        gradient=HEATMAP_GRADIENT
    ).add_to(m)

    # Property marker dots with popups
    name_col = 'property_name' if 'property_name' in df.columns \
               else ('property_id' if 'property_id' in df.columns else None)

    for _, row in df.iterrows():
        label    = row[name_col] if name_col else f'({row["latitude"]:.4f}, {row["longitude"]:.4f})'
        raw_val  = row[weight_col]
        norm_val = weights.loc[row.name]

        popup_rows = f'<b>{weight_col}:</b> {raw_val}<br>'
        # Show extra numeric columns in popup (skip coords and id)
        skip = {'latitude','longitude','property_id','property_name', weight_col}
        for col in df.select_dtypes(include='number').columns:
            if col not in skip:
                popup_rows += f'<b>{col}:</b> {row[col]}<br>'

        popup_html = f"""
        <div style="font-family:sans-serif;min-width:160px;">
            <b>{label}</b>
            <hr style="margin:4px 0">
            {popup_rows}
            <span style="color:#888;font-size:11px;">normalized weight: {norm_val:.3f}</span>
        </div>
        """
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=6,
            color='white',
            weight=1.5,
            fill=True,
            fill_color='#333',
            fill_opacity=0.85,
            popup=folium.Popup(popup_html, max_width=240)
        ).add_to(m)

    # Title overlay
    title_html = f"""
    <div style="position:fixed;top:12px;left:60px;z-index:9999;
                background:white;padding:10px 14px;border-radius:6px;
                border:2px solid #bbb;font-size:13px;
                box-shadow:2px 2px 8px rgba(0,0,0,.2);">
        <b>Edmonton Property Heatmap</b><br>
        <span style="color:#555;">Weight &nbsp;&#8594;&nbsp; <i>{weight_col}</i></span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))
    return m


if VISUALIZATION_MODE in ('heatmap', 'both'):
    print('Generating coordinate heatmap …')
    heatmap_map = create_coordinate_heatmap(df, WEIGHT_COLUMN)
    display(heatmap_map)
else:
    print('Skipping coordinate heatmap (set VISUALIZATION_MODE = "heatmap" or "both" to enable).')

---
## Step 4 – Edmonton Submarket Boundary Choropleth

Properties are assigned to submarkets via a **point-in-polygon** spatial join. Weights are then aggregated per submarket and rendered as a filled boundary map.

**Boundary source options** (set `BOUNDARY_SOURCE` below):

| Value | Description |
|-------|-------------|
| `'predefined'` | Built-in simplified Edmonton real-estate submarket polygons (no internet needed) |
| `'edmonton_api'` | Fetch Edmonton neighbourhood polygons from the City's Open Data portal |
| `'custom_file'` | Load your own GeoJSON file (set `CUSTOM_GEOJSON_PATH` and `GEOJSON_NAME_COL`) |

In [ ]:
# ── Boundary source selection ───────────────────────────────
BOUNDARY_SOURCE     = 'predefined'   # 'predefined' | 'edmonton_api' | 'custom_file'
CUSTOM_GEOJSON_PATH = None           # e.g. '/content/my_submarkets.geojson'
GEOJSON_NAME_COL    = 'submarket'    # column in your GeoJSON holding the area name


# ── Predefined Edmonton real-estate submarket polygons ──────
# Approximate bounding-box polygons covering major Edmonton investment areas.
# All coordinates are (longitude, latitude) as required by Shapely / GeoJSON.
PREDEFINED_SUBMARKETS = {
    'Downtown / Central': [
        (-113.530, 53.528), (-113.455, 53.528),
        (-113.455, 53.565), (-113.530, 53.565)
    ],
    'Oliver / West End': [
        (-113.572, 53.525), (-113.530, 53.525),
        (-113.530, 53.568), (-113.572, 53.568)
    ],
    'Glenora / Westmount': [
        (-113.588, 53.538), (-113.510, 53.538),
        (-113.510, 53.598), (-113.588, 53.598)
    ],
    'Strathcona / Whyte Ave': [
        (-113.538, 53.500), (-113.468, 53.500),
        (-113.468, 53.528), (-113.538, 53.528)
    ],
    'South Central': [
        (-113.568, 53.448), (-113.455, 53.448),
        (-113.455, 53.500), (-113.568, 53.500)
    ],
    'Southeast / Mill Woods': [
        (-113.472, 53.432), (-113.335, 53.432),
        (-113.335, 53.515), (-113.472, 53.515)
    ],
    'Southwest / Windermere': [
        (-113.675, 53.395), (-113.515, 53.395),
        (-113.515, 53.480), (-113.675, 53.480)
    ],
    'West Edmonton': [
        (-113.695, 53.488), (-113.572, 53.488),
        (-113.572, 53.555), (-113.695, 53.555)
    ],
    'North Central': [
        (-113.548, 53.562), (-113.448, 53.562),
        (-113.448, 53.628), (-113.548, 53.628)
    ],
    'Northeast / Clareview': [
        (-113.455, 53.562), (-113.335, 53.562),
        (-113.335, 53.658), (-113.455, 53.658)
    ],
    'Far North / Castle Downs': [
        (-113.582, 53.618), (-113.445, 53.618),
        (-113.445, 53.692), (-113.582, 53.692)
    ],
}


def build_predefined_gdf():
    records = [
        {'submarket': name,
         'geometry': Polygon(coords)}
        for name, coords in PREDEFINED_SUBMARKETS.items()
    ]
    gdf = gpd.GeoDataFrame(records, crs='EPSG:4326')
    print(f'Using {len(gdf)} predefined Edmonton submarkets.')
    return gdf, 'submarket'


def fetch_edmonton_api():
    """Try to pull Edmonton neighbourhood boundaries from the Open Data portal."""
    endpoints = [
        'https://data.edmonton.ca/resource/dkk9-cj3x.geojson',  # Neighbourhoods
        'https://data.edmonton.ca/resource/65fr-66s6.geojson',
    ]
    for url in endpoints:
        try:
            resp = requests.get(url, timeout=25)
            if resp.status_code == 200:
                gdf = gpd.GeoDataFrame.from_features(
                    resp.json()['features'], crs='EPSG:4326'
                )
                # find the most likely name column
                str_cols = gdf.select_dtypes('object').columns.tolist()
                name_col = next(
                    (c for c in str_cols if 'name' in c.lower() or 'neighbourhood' in c.lower()),
                    str_cols[0] if str_cols else 'name'
                )
                gdf = gdf.rename(columns={name_col: 'submarket'})
                print(f'Fetched {len(gdf)} neighbourhoods from Edmonton Open Data.')
                return gdf, 'submarket'
        except Exception as exc:
            print(f'Could not reach {url}: {exc}')
    print('Edmonton API unavailable — falling back to predefined submarkets.')
    return None, None


# ── Load boundaries ──────────────────────────────────────────
if BOUNDARY_SOURCE == 'edmonton_api':
    boundaries_gdf, area_name_col = fetch_edmonton_api()
    if boundaries_gdf is None:
        boundaries_gdf, area_name_col = build_predefined_gdf()

elif BOUNDARY_SOURCE == 'custom_file' and CUSTOM_GEOJSON_PATH:
    boundaries_gdf = gpd.read_file(CUSTOM_GEOJSON_PATH).to_crs('EPSG:4326')
    if GEOJSON_NAME_COL != 'submarket':
        boundaries_gdf = boundaries_gdf.rename(columns={GEOJSON_NAME_COL: 'submarket'})
    area_name_col = 'submarket'
    print(f'Loaded {len(boundaries_gdf)} areas from {CUSTOM_GEOJSON_PATH}.')

else:  # 'predefined'
    boundaries_gdf, area_name_col = build_predefined_gdf()

print(f'\nArea name column  : "{area_name_col}"')
print(f'Available areas   :')
for name in sorted(boundaries_gdf[area_name_col].tolist()):
    print(f'  • {name}')

In [ ]:
# ── Spatial join: assign each property to a submarket ───────
property_gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=[
        Point(lon, lat)
        for lat, lon in zip(df['latitude'], df['longitude'])
    ],
    crs='EPSG:4326'
)

joined = gpd.sjoin(
    property_gdf,
    boundaries_gdf[[area_name_col, 'geometry']],
    how='left',
    predicate='within'
).rename(columns={area_name_col: '_submarket'})

df_joined = pd.DataFrame(joined.drop(columns=['geometry', 'index_right'], errors='ignore'))

assigned   = df_joined['_submarket'].notna().sum()
unassigned = df_joined['_submarket'].isna().sum()
print(f'Properties assigned to a submarket : {assigned}')
print(f'Properties outside all boundaries  : {unassigned}')
if unassigned:
    print('  (unassigned properties appear on the heatmap but are omitted from the choropleth)')
    print('  Unassigned rows:')
    print(df_joined[df_joined['_submarket'].isna()][['property_name','latitude','longitude']].to_string(index=False))

In [ ]:
# ── Aggregate weight by submarket ───────────────────────────
agg_map = {'sum': 'sum', 'mean': 'mean', 'count': 'count'}
agg_fn  = agg_map.get(CHOROPLETH_AGGREGATION, 'sum')

if CHOROPLETH_AGGREGATION == 'count':
    submarket_agg = (
        df_joined.dropna(subset=['_submarket'])
        .groupby('_submarket')
        .size()
        .reset_index(name='agg_value')
    )
else:
    submarket_agg = (
        df_joined.dropna(subset=['_submarket'])
        .groupby('_submarket')[WEIGHT_COLUMN]
        .agg(agg_fn)
        .reset_index(name='agg_value')
    )

submarket_agg.columns = [area_name_col, 'agg_value']
print(f'Submarket aggregation  ({CHOROPLETH_AGGREGATION} of "{WEIGHT_COLUMN}"):')
print(submarket_agg.sort_values('agg_value', ascending=False).to_string(index=False))

---
## Step 5 – Choropleth Map

In [ ]:
def create_choropleth_map(boundaries_gdf, submarket_agg, area_name_col,
                          weight_col, center=MAP_CENTER, zoom=ZOOM_START):
    """
    Build a folium choropleth map of Edmonton submarket boundaries
    filled by aggregated weights.

    Parameters
    ----------
    boundaries_gdf : GeoDataFrame — submarket polygons
    submarket_agg  : DataFrame   — [area_name_col, 'agg_value']
    area_name_col  : str         — name column shared by both DataFrames
    weight_col     : str         — original weight column label (for legend)
    """
    m = folium.Map(location=center, zoom_start=zoom, tiles=MAP_TILE)

    # Merge agg values into GDF for tooltips
    merged = boundaries_gdf.merge(
        submarket_agg, on=area_name_col, how='left'
    )
    merged['agg_value'] = merged['agg_value'].fillna(0)

    agg_label = f'{CHOROPLETH_AGGREGATION.capitalize()} of {weight_col}'

    # Choropleth fill layer
    folium.Choropleth(
        geo_data=merged[[area_name_col, 'geometry']].to_json(),
        data=submarket_agg,
        columns=[area_name_col, 'agg_value'],
        key_on=f'feature.properties.{area_name_col}',
        fill_color=CHOROPLETH_COLOR,
        fill_opacity=0.72,
        line_opacity=0.6,
        line_color='#555',
        legend_name=agg_label,
        nan_fill_color='#e0e0e0',
        nan_fill_opacity=0.4
    ).add_to(m)

    # Invisible overlay for tooltip interactivity
    folium.GeoJson(
        merged.to_json(),
        style_function=lambda _: {
            'fillColor': 'transparent',
            'color':     'transparent'
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[area_name_col, 'agg_value'],
            aliases=['Submarket', agg_label],
            localize=True,
            sticky=True
        )
    ).add_to(m)

    # Property dots
    name_col = 'property_name' if 'property_name' in df.columns \
               else ('property_id' if 'property_id' in df.columns else None)
    for _, row in df.iterrows():
        label = row[name_col] if name_col else ''
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=5,
            color='white',
            weight=1.5,
            fill=True,
            fill_color='#444',
            fill_opacity=0.9,
            popup=folium.Popup(
                f'<b>{label}</b><br>{weight_col}: {row[weight_col]}',
                max_width=200
            )
        ).add_to(m)

    # Title
    title_html = f"""
    <div style="position:fixed;top:12px;left:60px;z-index:9999;
                background:white;padding:10px 14px;border-radius:6px;
                border:2px solid #bbb;font-size:13px;
                box-shadow:2px 2px 8px rgba(0,0,0,.2);">
        <b>Edmonton Submarket Choropleth</b><br>
        <span style="color:#555;">{agg_label}</span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))
    return m


if VISUALIZATION_MODE in ('choropleth', 'both'):
    print('Generating submarket choropleth …')
    choropleth_map = create_choropleth_map(
        boundaries_gdf, submarket_agg, area_name_col, WEIGHT_COLUMN
    )
    display(choropleth_map)
else:
    print('Skipping choropleth (set VISUALIZATION_MODE = "choropleth" or "both" to enable).')

---
## Step 6 – Combined Map

Overlays the heatmap with the submarket boundary outlines on a single map. Click any property dot for details; hover boundaries for submarket labels.

In [ ]:
def create_combined_map(df, boundaries_gdf, weight_col, area_name_col,
                         center=MAP_CENTER, zoom=ZOOM_START):
    """
    Combines the heatmap layer with submarket boundary outlines.

    Parameters
    ----------
    df             : DataFrame with property lat/lon and weight_col
    boundaries_gdf : GeoDataFrame of submarket polygons
    weight_col     : column used for heatmap intensity
    area_name_col  : column in boundaries_gdf holding area names
    """
    m = folium.Map(location=center, zoom_start=zoom, tiles=MAP_TILE)

    # Heatmap layer
    weights   = normalize_weights(df[weight_col], NORMALIZE_METHOD)
    heat_data = list(zip(df['latitude'], df['longitude'], weights))
    HeatMap(
        heat_data,
        radius=HEATMAP_RADIUS,
        blur=HEATMAP_BLUR,
        min_opacity=HEATMAP_MIN_OPACITY,
        gradient=HEATMAP_GRADIENT,
        name='Heat Layer'
    ).add_to(m)

    # Submarket boundary outlines
    merged = boundaries_gdf.merge(
        submarket_agg, on=area_name_col, how='left'
    )
    merged['agg_value'] = merged['agg_value'].fillna(0)
    agg_label = f'{CHOROPLETH_AGGREGATION.capitalize()} of {weight_col}'

    folium.GeoJson(
        merged.to_json(),
        style_function=lambda _: {
            'fillColor': 'transparent',
            'color':     '#333',
            'weight':    2.5,
            'fillOpacity': 0
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[area_name_col, 'agg_value'],
            aliases=['Submarket', agg_label],
            localize=True,
            sticky=True
        ),
        name='Submarket Boundaries'
    ).add_to(m)

    # Property markers
    name_col = 'property_name' if 'property_name' in df.columns \
               else ('property_id' if 'property_id' in df.columns else None)
    for _, row in df.iterrows():
        label    = row[name_col] if name_col else ''
        norm_val = weights.loc[row.name]
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=6,
            color='white',
            weight=1.5,
            fill=True,
            fill_color='#333',
            fill_opacity=0.9,
            popup=folium.Popup(
                f'<b>{label}</b><br>{weight_col}: {row[weight_col]}<br>'
                f'<span style="color:#888;">norm: {norm_val:.3f}</span>',
                max_width=220
            )
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)

    # Title
    title_html = f"""
    <div style="position:fixed;top:12px;left:60px;z-index:9999;
                background:white;padding:10px 14px;border-radius:6px;
                border:2px solid #bbb;font-size:13px;
                box-shadow:2px 2px 8px rgba(0,0,0,.2);">
        <b>Edmonton Heatmap + Submarkets</b><br>
        <span style="color:#555;">Weight &nbsp;&#8594;&nbsp; <i>{weight_col}</i></span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(title_html))
    return m


if VISUALIZATION_MODE == 'both':
    print('Generating combined map …')
    combined_map = create_combined_map(
        df, boundaries_gdf, WEIGHT_COLUMN, area_name_col
    )
    display(combined_map)
else:
    print('Skipping combined map (set VISUALIZATION_MODE = "both" to enable).')

---
## Step 7 – Summary Table & Bar Chart (Optional)

Tabular summary of aggregated values per submarket and a Plotly bar chart for quick comparison.

In [ ]:
# ── Full summary table ───────────────────────────────────────
if CHOROPLETH_AGGREGATION == 'count':
    summary = df_joined.dropna(subset=['_submarket']).groupby('_submarket').agg(
        property_count=('latitude', 'count')
    ).reset_index().rename(columns={'_submarket': 'Submarket'})
else:
    summary = df_joined.dropna(subset=['_submarket']).groupby('_submarket').agg(
        property_count=('latitude', 'count'),
        weight_sum=(WEIGHT_COLUMN, 'sum'),
        weight_mean=(WEIGHT_COLUMN, 'mean'),
        weight_min=(WEIGHT_COLUMN, 'min'),
        weight_max=(WEIGHT_COLUMN, 'max')
    ).reset_index().rename(columns={
        '_submarket': 'Submarket',
        'weight_sum':  f'sum({WEIGHT_COLUMN})',
        'weight_mean': f'mean({WEIGHT_COLUMN})',
        'weight_min':  f'min({WEIGHT_COLUMN})',
        'weight_max':  f'max({WEIGHT_COLUMN})'
    })

summary = summary.sort_values('property_count', ascending=False)
print(summary.to_string(index=False))

In [ ]:
# ── Plotly bar chart ─────────────────────────────────────────
agg_label = f'{CHOROPLETH_AGGREGATION.capitalize()} of {WEIGHT_COLUMN}'

fig = px.bar(
    submarket_agg.sort_values('agg_value', ascending=True),
    x='agg_value',
    y=area_name_col,
    orientation='h',
    labels={'agg_value': agg_label, area_name_col: 'Submarket'},
    title=f'Edmonton Submarkets — {agg_label}',
    color='agg_value',
    color_continuous_scale='YlOrRd',
    template='plotly_white'
)
fig.update_layout(
    height=max(350, len(submarket_agg) * 38),
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'}
)
fig.show()

---
## Step 8 – Export Maps (Optional)

Save any map as a standalone HTML file that can be shared or embedded.

In [ ]:
# ── Save maps to HTML ────────────────────────────────────────
# Uncomment whichever maps you want to export.

# heatmap_map.save('edmonton_heatmap.html')
# print('Saved: edmonton_heatmap.html')

# choropleth_map.save('edmonton_choropleth.html')
# print('Saved: edmonton_choropleth.html')

# combined_map.save('edmonton_combined.html')
# print('Saved: edmonton_combined.html')

# ── Download from Colab ──────────────────────────────────────
# from google.colab import files
# files.download('edmonton_combined.html')

print('Uncomment the lines above to save / download map files.')

---
## Quick Reference — Switching Weight Columns

| Goal | Set `WEIGHT_COLUMN` to |
|------|------------------------|
| Total unit counts | `'units'` |
| Physical vacancy rate | `'vacancy_rate'` |
| Rent per square foot | `'rent_psf'` |
| Absorption rate | `'absorption_rate'` |
| Any custom ratio column | `'your_column_name'` |

Also consider pairing with `CHOROPLETH_AGGREGATION = 'mean'` for rate/ratio columns
and `'sum'` for count-based columns like `units`.

---
*Built with [folium](https://python-visualization.github.io/folium/) · [GeoPandas](https://geopandas.org/) · [Plotly](https://plotly.com/python/)*